# 动态规划算法

## 简介
动态规划（Dynamic Programming）是程序设计算法中非常重要的内容，能够高效解决一些经典问题，例如背包问题和最短路径规划。动态规划的基本思想是将待求解问题分解成若干个子问题，先求解子问题，然后从这些子问题的解得到原问题的解。在动态规划中，我们会保存已解决的子问题的答案，而在求解目标问题过程中，如果需要这些子问题答案时，就可以直接利用，避免重复计算。在本节课程内容中，我们就学习如何用动态规划的思想来求解在有限马尔可夫决策过程中的最优策略。

基于动态规划的强化学习算法主要有两种：一是策略迭代（Policy Iteration），二是价值迭代(Value Iteration)。其中，策略迭代有两部分组成：策略评估（Policy Evaluation）和策略提升（Policy Improvement）。具体来说，策略迭代中的策略评估使用贝尔曼期望方程来得到一个策略的状态价值函数，这是一个动态规划的过程。而价值迭代直接使用贝尔曼最优方程来进行动态规划，得到最终的最优状态价值。

不同于我们之前介绍的蒙特卡洛算法和之后将要介绍的时序差分算法，基于动态规划的两大强化学习算法要求我们能事先知道环境的状态转移函数和奖励函数。在这样一个白盒环境中，我们不需要通过智能体和环境的大量交互来学习，可以直接用动态规划求解状态价值。但是，现实中白盒环境很少，这也是动态规划算法的局限所在，我们无法将其运用到很多实际场景中。其次，我们介绍的策略迭代和价值迭代通常只适用于有限马尔可夫决策过程中，即状态空间和动作空间是有限的。

## Cliff Walking悬崖漫步环境介绍
接下来一起来看一看Cliff Walk环境的代码吧。

In [1]:
# 导包
import copy


# 定义Cliff Walking环境类
class CliffWalkingEnv:
    """Cliff Walking环境"""
    def __init__(self, ncol = 12, nrow = 4):
        self.ncol = ncol        # 定义环境的宽
        self.nrow = nrow        # 定义环境的高
        # 转移矩阵P[state][action] = [(p, next_state, reward, done)]，包含下一个状态和奖励
        self.P = self.createP()

    def createP(self):
        """构建状态转移矩阵P"""
        # 初始化
        P = [[[] for j in range(4)] for i in range(self.nrow * self.ncol)]      # 构建一个48行4列的空列表
        change = [[0, -1], [0, 1], [-1, 0], [1, 0]]     # 4种动作，0:上，1:下，2:左，3:右。原点(0, 0)定义在左上角，但起点在左下角
        for i in range(self.nrow):          # 遍历每行
            for j in range(self.ncol):      # 遍历每列
                for a in range(4):
                    # i * self.ncol + j：把二维坐标（第几行第几列）一维化，方便存入数组
                    # 比如 i=3, j=5（第4行第6列，在悬崖里），公式就是 3 * 12 + 5 = 41，这个状态编号就是 41
                    # [a]：当前采取的某个动作（0上、1下、2左、3右）
                    # 强化学习环境的标准格式 [(转移概率 p, 下一个状态 s', 奖励 r, 是否结束 done)]
                    # p = 1说明一定会转移，这是环境决定的
                    # s' = i * self.ncol + j其实就是留在原地了
                    # r = 0，因为游戏结束了，没有奖励
                    # done = True即游戏结束标志
                    if i == self.nrow - 1 and j > 0:        # 位置在悬崖或者终点，因为无法继续交互，任何动作奖励都为0
                        P[i * self.ncol + j][a] = [(1, i * self.ncol + j - 1, 0, True)]
                        continue

                    # 其他位置
                    # min 和 max 保证了坐标绝不会超出 0~11（列）和 0~3（行）的范围。
                    next_x = min(self.ncol - 1, max(0, j + change[a][0]))       # 计算下一步的位置的x，处理撞墙边界问题。
                    next_y = min(self.nrow - 1, max(0, i + change[a][1]))       # 计算下一步的位置的y，处理撞墙边界问题。
                    next_state = next_y * self.ncol + next_x        # 把二维坐标 (next_y, next_x) 转换成一维的编号。公式同上
                    reward = -1             # 默认每走一步，由于消耗时间，奖励都是 -1
                    done = False            # 默认走一步还没结束
                    if next_y == self.nrow - 1 and next_x > 0:      # 下一步在最后一行且不在起点，即下一个位置在悬崖或者终点
                        done = True             # 游戏怎么样都会结束
                        if next_x != self.ncol - 1:         # 下一个位置不是终点，即在悬崖
                            reward = -100       # 掉入悬崖，奖励为-100
                    P[i * self.ncol + j][a] = [(1, next_state, reward, done)]
        return P

## 策略迭代
我们现在来看一下策略迭代的代码。

In [3]:
# 定义策略迭代类
class PolicyIteration:
    """策略迭代算法"""
    # 参2：环境实例（例如悬崖漫步）；参3：策略评估收敛阈值；参4：折扣因子gamma
    def __init__(self, env, theta, gamma):
        self.env = env
        # 状态价值向量V：这里初始化了一个长度为 48（12列 × 4行）的全0一维数组
        # 物理意义：智能体在开局时，对每个“城市（状态）”的长期价值一无所知，所以它的“价值账本”全是 0。
        self.v = [0] * self.env.ncol * self.env.nrow

        # 这是一个 48×4 的矩阵。48 个状态，每个状态有 4 个动作（上、下、左、右），每个动作对应一个概率，这里初始化为随机均匀策略。
        self.pi = [[0.25, 0.25, 0.25, 0.25] for i in range(self.env.ncol * self.env.nrow)]
        self.theta = theta      # 策略评估收敛阈值（其实就是eps误差阈值）
        self.gamma = gamma      # 折扣因子

    def policy_evaluation(self):        # 策略评估
        cnt = 1         # 计数器，记录迭代次数
        while 1:        # 幂迭代法，无限循环，直到收敛才 break
            max_diff =0         # 记录本轮和上一轮 V 值的最大误差
            new_v = [0] * self.env.ncol * self.env.nrow     # 准备一个空的新账本，存这一轮算出来的 V 值
            for s in range(self.env.ncol * self.env.nrow):      # 遍历这48个格子（以悬崖漫步为例）
                qsa_list = []           # 实则存的是状态价值的分量
                for a in range(4):      # 遍历4个动作（对应向4个方向走）
                    qsa = 0
                    for res in self.env.P[s][a]:        # 遍历转移元组：P[s][a] = [(p, next_state, r, done)]
                        p, next_state, r, done = res
                        # 这里用到的其实就是贝尔曼期望方程中的动作价值函数
                        # 本章环境比较特殊，奖励和下一状态有关，所以需要和状态转移概率相乘
                        # 乘以(1 - done)：如果 done=True，(1-1)=0，未来价值直接归零；如果 done=False，(1-0)=1，正常累加未来价值。
                        qsa += p * (r + self.gamma * self.v[next_state] * (1 - done))       # 到终点了未来价值就清零了
                    qsa_list.append(self.pi[s][a] * qsa)
                new_v[s] = sum(qsa_list)        # 状态价值函数和动作价值函数之间的关系：状态价值就是动作价值的加权平均（期望）
                max_diff = max(max_diff, abs(new_v[s] - self.v[s]))     # 将本轮和上轮 V 值的误差与当前最大误差的较大值作为最大误差
            # 在下一轮 while 循环开始时，self.v 就是刚算出的最新价值。而在下一轮里，代码会继续利用这个最新价值去算动作价值 Q 值。
            self.v = new_v
            if max_diff < self.theta:       # 满足收敛条件后就退出迭代
                break
            cnt += 1        # 每迭代一次就更新一次计数器
        print(f"策略评估在进行{cnt}轮后完成")

    def policy_improvement(self):       # 策略提升
        for s in range(self.env.nrow * self.env.ncol):      # 依然是遍历这 48 个格子
            qsa_list = []           # 用于记录当前格子下的4个动作各自的动作价值
            for a in range(4):      # 遍历4个动作（对应向4个方向走）
                qsa = 0
                for res in self.env.P[s][a]:
                    p, next_state, r, done = res
                    qsa += p * (r + self.gamma * self.v[next_state] * (1 - done))
                qsa_list.append(qsa)        # 将动作价值加入列表
            maxq = max(qsa_list)        # 找出当前格子下最大的动作价值
            cntq = qsa_list.count(maxq)     # 计算有几个动作得到了最大的Q值
            # 让得到了最大 Q 值的动作均分概率，即贪心策略，让智能体在状态 s 不再盲目乱走，而是：直奔那个能让 Q 值最大的动作！
            # 将这些动作的概率设为 1 / cntq，其他动作的概率设为 0，保证智能体只会向当前已知 Q 值最高的方向探索，不再盲目乱走。
            self.pi[s] = [1 / cntq if q == maxq else 0 for q in qsa_list]
        print("策略提升完成")
        return self.pi      # 返回更新后的策略

    def policy_iteration(self):         # 策略迭代
        while 1:
            self.policy_evaluation()        # 执行策略评估
            old_pi = copy.deepcopy(self.pi)         # 将列表进行深拷贝，便于接下来进行比较
            new_pi = self.policy_improvement()        # 执行策略提升，并将新策略赋值给 new_pi
            if old_pi == new_pi:      # 如果策略没有更新，则表示策略迭代完成
                break

现在我们已经写好了环境代码和策略迭代代码。为了更好的展现最终的策略，我们下面增加一个打印策略的函数，来打印当前策略每个状态下的价值以及会采取的动作。对于打印出来的动作，我们用"^o<o"表示等概率采取向左和向上两种动作，"ooo>"表示在当前状态只采取向右动作。

In [4]:
# 参1：智能体（环境实体）；参2：动作含义；参3：悬崖位置；参4：终点位置
def print_agent(agent, action_meaning, disaster = [], end = []):
    print("状态价值：")
    for i in range(agent.env.nrow):
        for j in range(agent.env.ncol):
            # 为了输出美观，保持输出6个字符
            print('%6.6s' % ('%.3f' % agent.v[i * agent.env.ncol + j]), end = ' ')
        print()

    print("策略：")
    for i in range(agent.env.nrow):
        for j in range(agent.env.ncol):
            # 一些特殊状态，如悬崖漫步中的悬崖
            if (i * agent.env.ncol + j) in disaster:
                print("****", end = ' ')
            elif (i * agent.env.ncol + j) in end:
                print("EEEE", end = ' ')
            else:
                a = agent.pi[i * agent.env.ncol + j]
                pi_str = ''
                for k in range(len(action_meaning)):
                    pi_str += action_meaning[k] if a[k] > 0 else 'o'
                print(pi_str, end = ' ')
        print()


# 测试
env = CliffWalkingEnv()
action_meaning = ['^', 'v', '<', '>']
theta = 0.001
gamma = 0.9
agent = PolicyIteration(env, theta, gamma)
agent.policy_iteration()
print_agent(agent, action_meaning, list(range(37, 47)), [47])

策略评估在进行60轮后完成
策略提升完成
策略评估在进行72轮后完成
策略提升完成
策略评估在进行44轮后完成
策略提升完成
策略评估在进行12轮后完成
策略提升完成
策略评估在进行1轮后完成
策略提升完成
状态价值：
-7.712 -7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 
-7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 
-7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 -1.000 
-7.458  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000 
策略：
ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo 
ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo 
ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ovoo 
^ooo **** **** **** **** **** **** **** **** **** **** EEEE 


我们发现经过五次策略评估和策略提升的循环迭代，策略收敛了。此时获得的策略也已经打印出来了，我们可以用贝尔曼最优方程去检验每一个状态的价值，我们可以发现最终输出的策略的确是最有策略。

## 价值迭代
我们现在来实现价值迭代的代码。

In [8]:
# 定义价值迭代类
class ValueIteration:
    """价值迭代算法"""
    # 参2：环境实例（例如悬崖漫步）；参3：收敛阈值；参4：折扣因子gamma
    def __init__(self, env, theta, gamma):
        self.env = env
        # 状态价值向量V：长度为 48（12列 × 4行）的全0一维数组。
        # 物理意义：智能体对每个状态的长期价值一无所知，初始价值账本全是 0。
        self.v = [0] * self.env.ncol * self.env.nrow

        self.theta = theta          # 价值收敛阈值（误差eps）。两轮迭代之间价值变化小于此值，说明账本稳定，即可停止。
        self.gamma = gamma          # 折扣因子，决定了未来奖励在当前眼中的分量。

        # 【价值迭代的特点】：初始化时不需要固定的策略矩阵（不像策略迭代里有初始的 0.25 矩阵）。
        # 这里只是先准备一个空列表（长度为48），等价值全部迭代收敛后，再根据最终的 V 值“一把抽出”最终策略。
        self.pi = [None for i in range(self.env.ncol * self.env.nrow)]

    def value_iteration(self):      # 核心大循环（这是唯一的一层大循环，没有“评估”和“提升”的嵌套）
        cnt = 0         # 计数器，记录迭代轮数
        while 1:        # 无限循环，直到价值收敛才 break
            max_diff = 0         # 记录本轮和上一轮 V 值的最大误差
            new_v = [0] * self.env.ncol * self.env.nrow     # 准备一个新的空账本，存这一轮算出来的 V 值

            for s in range(self.env.ncol * self.env.nrow):      # 遍历这48个格子（状态）
                qsa_list = []           # 用于存储当前状态 s 下，采取4个动作各自的 Q 值

                for a in range(4):      # 遍历4个动作（对应向4个方向走）
                    qsa = 0
                    # 遍历转移元组：P[s][a] = [(p, next_state, r, done)]
                    for res in self.env.P[s][a]:
                        p, next_state, r, done = res
                        # 【核心公式】这里是算动作价值函数 Q(s,a) 的期望值
                        # 本章环境比较特殊，奖励和下一状态有关，所以需要和状态转移概率相乘
                        # 乘以(1 - done)：如果 done=True，(1-1)=0，未来价值直接归零；如果 done=False，(1-0)=1，正常累加未来价值。
                        qsa += p * (r + self.gamma * self.v[next_state] * (1 - done))
                    qsa_list.append(qsa)        # 把算出的这个动作的 Q 值存进列表

                # 【关键区别1】：价值迭代的核心：不求期望（不乘以策略概率），而是直接“贪心”。
                # 挑出 qsa_list 里最大的 Q 值，直接作为当前状态 s 的最新状态价值 V(s)。
                new_v[s] = max(qsa_list)
                # 比较刚算出的新价值和旧账本里的旧价值，记录最大误差
                max_diff = max(max_diff, abs(new_v[s] - self.v[s]))

            # 在下一轮 while 循环开始前，用新账本覆盖旧账本，继续下一轮迭代
            self.v = new_v
            if max_diff < self.theta:       # 当所有状态的价值更新幅度都极小了，说明 V 已经收敛到最优
                break
            cnt += 1        # 每迭代一次就更新一次计数器

        print(f"价值迭代一共进行{cnt}轮")
        # 循环结束，意味着拿到了最优价值函数 V*。现在根据 V* 提取最终的策略
        self.get_policy()

    def get_policy(self):       # 策略提取（相当于策略迭代中的“策略提升”）
        for s in range(self.env.ncol * self.env.nrow):      # 遍历这 48 个格子
            qsa_list = []           # 用于记录当前格子下的 4 个动作各自的 Q 值
            for a in range(4):      # 遍历 4 个动作
                qsa = 0
                for res in self.env.P[s][a]:
                    p, next_state, r, done = res
                    qsa += p * (r + self.gamma * self.v[next_state] * (1 - done))
                qsa_list.append(qsa)        # 将动作价值加入列表

            # 找出当前格子下，最大的动作价值
            maxq = max(qsa_list)
            cntq = qsa_list.count(maxq)     # 计算有几个动作得到了最大的 Q 值（处理并列第一的情况）

            # 【关键区别2】：因为价值迭代过程中没有显式的策略矩阵，所以在这里一次性生成策略。
            # 让得到了最大 Q 值的动作均分概率，其他动作概率为 0。
            # 这就是最终的确定性贪心策略。例如 [0, 1.0, 0, 0] 表示 100% 采取某个动作。
            self.pi[s] = [1 / cntq if q == maxq else 0 for q in qsa_list]


# 测试
env = CliffWalkingEnv()
action_meaning = ['^', 'v', '<', '>']
theta = 0.001
gamma = 0.9
agent = ValueIteration(env, theta, gamma)
agent.value_iteration()
print_agent(agent, action_meaning, list(range(37, 47)), [47])

价值迭代一共进行14轮
状态价值：
-7.712 -7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 
-7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 
-7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 -1.000 
-7.458  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000 
策略：
ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo 
ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo 
ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ovoo 
^ooo **** **** **** **** **** **** **** **** **** **** EEEE 


价值迭代总共用了14轮而策略迭代中策略评估总共用了263轮，可以发现价值迭代中循环次数远少于策略迭代。

## Frozen Lake环境
暂时跳过。